# Notebook 00 - Corpus calendario Mundial 2026

Lee `Corpus_Mundial/calendario-2026.json` (104 partidos) y genera 1 archivo `.md` por partido en `Corpus_Mundial/calendario-2026/`. Frontmatter YAML + cuerpo en espanol.

## 1. Setup

In [2]:
from __future__ import annotations

import json
import re
import unicodedata
from datetime import date
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
JSON_PATH = PROJECT_ROOT / "Corpus_Mundial" / "calendario-2026.json"
OUTPUT_DIR = PROJECT_ROOT / "Corpus_Mundial" / "calendario-2026"

print(f"JSON input:  {JSON_PATH} (existe: {JSON_PATH.exists()})")
print(f"Output dir:  {OUTPUT_DIR}")

JSON input:  c:\Users\Administrador\Documents\MaestriaUSFQ\Procesamiento Lenguaje Natural\Proyecto Final\Corpus_Mundial\calendario-2026.json (existe: True)
Output dir:  c:\Users\Administrador\Documents\MaestriaUSFQ\Procesamiento Lenguaje Natural\Proyecto Final\Corpus_Mundial\calendario-2026


## 2. Diccionarios de traduccion

Equipos y fases vienen en ingles en el JSON. Traducir a espanol para que las queries matcheen.

In [3]:
TEAM_ES = {
    "Algeria": "Argelia",
    "Argentina": "Argentina",
    "Australia": "Australia",
    "Austria": "Austria",
    "Belgium": "Belgica",
    "Bosnia-Herzegovina": "Bosnia y Herzegovina",
    "Brazil": "Brasil",
    "Cabo Verde": "Cabo Verde",
    "Canada": "Canada",
    "Colombia": "Colombia",
    "Congo DR": "Republica Democratica del Congo",
    "Croatia": "Croacia",
    "Curaçao": "Curazao",
    "Czechia": "Republica Checa",
    "Côte d'Ivoire": "Costa de Marfil",
    "Ecuador": "Ecuador",
    "Egypt": "Egipto",
    "England": "Inglaterra",
    "France": "Francia",
    "Germany": "Alemania",
    "Ghana": "Ghana",
    "Haiti": "Haiti",
    "IR Iran": "Iran",
    "Iraq": "Irak",
    "Japan": "Japon",
    "Jordan": "Jordania",
    "Korea Republic": "Corea del Sur",
    "Mexico": "Mexico",
    "Morocco": "Marruecos",
    "Netherlands": "Paises Bajos",
    "New Zealand": "Nueva Zelanda",
    "Norway": "Noruega",
    "Panama": "Panama",
    "Paraguay": "Paraguay",
    "Por definir": "Por definir",
    "Portugal": "Portugal",
    "Qatar": "Catar",
    "Saudi Arabia": "Arabia Saudita",
    "Scotland": "Escocia",
    "Senegal": "Senegal",
    "South Africa": "Sudafrica",
    "Spain": "España",
    "Sweden": "Suecia",
    "Switzerland": "Suiza",
    "Tunisia": "Tunez",
    "Turkey": "Turquia",
    "USA": "Estados Unidos",
    "Uruguay": "Uruguay",
    "Uzbekistan": "Uzbekistan",
}

FASE_ES = {
    "group": "Fase de Grupos",
    "R32": "Dieciseisavos de Final",
    "R16": "Octavos de Final",
    "QF": "Cuartos de Final",
    "SF": "Semifinales",
    "3rd": "Tercer Puesto",
    "final": "Final",
}

DIAS_SEMANA = ["lunes", "martes", "miercoles", "jueves", "viernes", "sabado", "domingo"]
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio",
         "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]


def team_es(name):
    return TEAM_ES.get(name, name)


def fase_es(code):
    return FASE_ES.get(code, code)


def fecha_legible(iso_date):
    # 2026-06-11 -> 'jueves 11 de junio de 2026'
    d = date.fromisoformat(iso_date)
    return f"{DIAS_SEMANA[d.weekday()]} {d.day} de {MESES[d.month - 1]} de {d.year}"


# Test rapido
print(team_es("USA"), "|", team_es("Korea Republic"), "|", team_es("Por definir"))
print(fase_es("group"), "|", fase_es("QF"), "|", fase_es("final"))
print(fecha_legible("2026-06-11"))

Estados Unidos | Corea del Sur | Por definir
Fase de Grupos | Cuartos de Final | Final
jueves 11 de junio de 2026


## 3. Slug ASCII-safe para filenames

In [4]:
def slugify(text):
    # Convierte a ASCII lowercase con guiones. Quita acentos, espacios, caracteres especiales.
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "-", text)
    text = text.strip("-")
    return text


# Test
for t in ["México", "Korea Republic", "Por definir", "Côte d'Ivoire", "USA"]:
    print(f"{t!r:25s} -> {slugify(t)}")

'México'                  -> mexico
'Korea Republic'          -> korea-republic
'Por definir'             -> por-definir
"Côte d'Ivoire"           -> cote-d-ivoire
'USA'                     -> usa


## 4. Plantilla del documento

Cada partido: frontmatter YAML (metadata Chroma) + cuerpo en espanol con la info repetida en lista y parrafo.

In [5]:
def match_to_markdown(m):
    # m: dict del JSON con las claves originales.
    pid = m["match_id"]
    fase = m["fase"]
    fase_legible = fase_es(fase)
    grupo = m.get("grupo")
    local_orig = m["equipo_local"]
    visit_orig = m["equipo_visitante"]
    local = team_es(local_orig)
    visit = team_es(visit_orig)
    fecha = m["fecha"]
    fecha_dia = fecha_legible(fecha)
    hora_local = m["hora_local"]
    hora_utc = m["hora_utc"]
    estadio = m["estadio"]
    ciudad = m["ciudad"]
    pais = m["pais"]

    # Titulo segun fase
    if fase == "group":
        titulo = f"{local} vs {visit} — {fase_legible}, Grupo {grupo}"
    elif fase in ("R32", "R16", "QF", "SF"):
        titulo = f"Partido {pid} — {fase_legible}"
    elif fase == "final":
        titulo = f"Final del Mundial 2026"
    elif fase == "3rd":
        titulo = f"Partido por el Tercer Puesto — Mundial 2026"
    else:
        titulo = f"Partido {pid} — Mundial 2026"

    # Tags
    tags = ["mundial-2026", "calendario", "partido", fase, slugify(local), slugify(visit), slugify(estadio)]
    if grupo:
        tags.append(f"grupo-{grupo.lower()}")
    tags = [t for t in tags if t and t != "por-definir"]

    # Frontmatter YAML
    fm_lines = [
        "---",
        f"titulo: {titulo}",
        f"tema: calendario-mundial-2026",
        f"tipo: partido",
        f"fuente: Plataforma 91 / FIFA",
        f"partido_id: {pid}",
        f"edicion: 2026",
        f"fase: {fase}",
        f"fase_es: {fase_legible}",
    ]
    if grupo:
        fm_lines.append(f"grupo: {grupo}")
    fm_lines.extend([
        f"fecha: {fecha}",
        f"hora_local: '{hora_local}'",
        f"hora_utc: '{hora_utc}'",
        f"estadio: {estadio}",
        f"ciudad: {ciudad}",
        f"pais: {pais}",
        f"equipo_local: {local}",
        f"equipo_visitante: {visit}",
        f"equipo_local_original: {local_orig}",
        f"equipo_visitante_original: {visit_orig}",
        "tags:",
    ])
    for t in tags:
        fm_lines.append(f"  - {t}")
    fm_lines.append("---")
    frontmatter = "\n".join(fm_lines)

    # Cuerpo del doc
    grupo_text = f", Grupo {grupo}" if grupo else ""
    cuerpo_lines = [
        f"# {titulo}",
        "",
        f"Partido número **{pid}** del Mundial 2026.",
        "",
        f"**Fase:** {fase_legible}{grupo_text}",
        f"**Fecha:** {fecha_dia} ({fecha})",
        f"**Hora local:** {hora_local} (hora del estadio)",
        f"**Hora UTC:** {hora_utc}",
        f"**Estadio:** {estadio}",
        f"**Ciudad:** {ciudad}, {pais}",
        f"**Equipo local:** {local}",
        f"**Equipo visitante:** {visit}",
        "",
        "## Descripcion",
        "",
    ]
    if fase == "group" and local != "Por definir" and visit != "Por definir":
        cuerpo_lines.append(
            f"{local} enfrenta a {visit} en la {fase_legible.lower()} del Mundial 2026, "
            f"en el Grupo {grupo}. El partido se disputa el {fecha_dia} a las {hora_local} hora local "
            f"({hora_utc} UTC) en el {estadio} de {ciudad}, {pais}."
        )
    elif fase == "final":
        cuerpo_lines.append(
            f"La gran final del Mundial 2026 se disputa el {fecha_dia} a las {hora_local} hora local "
            f"({hora_utc} UTC) en el {estadio} de {ciudad}, {pais}. "
            f"Es el partido decisivo del torneo, el numero {pid} del calendario completo."
        )
    elif fase == "3rd":
        cuerpo_lines.append(
            f"El partido por el tercer puesto del Mundial 2026 se disputa el {fecha_dia} "
            f"a las {hora_local} hora local ({hora_utc} UTC) en el {estadio} de {ciudad}, {pais}."
        )
    else:
        cuerpo_lines.append(
            f"Partido de {fase_legible.lower()} del Mundial 2026. Se disputa el {fecha_dia} "
            f"a las {hora_local} hora local ({hora_utc} UTC) en el {estadio} de {ciudad}, {pais}. "
            f"Los equipos clasificados a esta fase se determinan en rondas previas."
        )

    cuerpo = "\n".join(cuerpo_lines)
    return frontmatter + "\n\n" + cuerpo + "\n"


# Test con el primer partido del JSON
with open(JSON_PATH, encoding="utf-8") as f:
    matches = json.load(f)

print(match_to_markdown(matches[0]))
print("=" * 60)
print(match_to_markdown(matches[-1]))  # ultimo partido (probable la final)

---
titulo: Mexico vs Sudafrica — Fase de Grupos, Grupo A
tema: calendario-mundial-2026
tipo: partido
fuente: Plataforma 91 / FIFA
partido_id: 1
edicion: 2026
fase: group
fase_es: Fase de Grupos
grupo: A
fecha: 2026-06-11
hora_local: '14:00'
hora_utc: '19:00'
estadio: Estadio Azteca
ciudad: Mexico City
pais: Mexico
equipo_local: Mexico
equipo_visitante: Sudafrica
equipo_local_original: Mexico
equipo_visitante_original: South Africa
tags:
  - mundial-2026
  - calendario
  - partido
  - group
  - mexico
  - sudafrica
  - estadio-azteca
  - grupo-a
---

# Mexico vs Sudafrica — Fase de Grupos, Grupo A

Partido número **1** del Mundial 2026.

**Fase:** Fase de Grupos, Grupo A
**Fecha:** jueves 11 de junio de 2026 (2026-06-11)
**Hora local:** 14:00 (hora del estadio)
**Hora UTC:** 19:00
**Estadio:** Estadio Azteca
**Ciudad:** Mexico City, Mexico
**Equipo local:** Mexico
**Equipo visitante:** Sudafrica

## Descripcion

Mexico enfrenta a Sudafrica en la fase de grupos del Mundial 2026, en el G

## 5. Generar todos los archivos

In [6]:
def filename_for(m):
    pid = m["match_id"]
    fase = m["fase"]
    local = team_es(m["equipo_local"])
    visit = team_es(m["equipo_visitante"])
    if local == "Por definir" and visit == "Por definir":
        # KO sin equipos asignados: usar fase + match_id
        return f"partido-{pid:03d}-{slugify(fase_es(fase))}.md"
    return f"partido-{pid:03d}-{slugify(local)}-vs-{slugify(visit)}.md"


OUTPUT_DIR.mkdir(exist_ok=True)

# Limpiar archivos previos para idempotencia (solo .md, no la carpeta)
for prev in OUTPUT_DIR.glob("*.md"):
    prev.unlink()
print(f"Output dir limpio: {OUTPUT_DIR}")

from collections import Counter
fase_counter = Counter()

for m in matches:
    fname = filename_for(m)
    content = match_to_markdown(m)
    (OUTPUT_DIR / fname).write_text(content, encoding="utf-8")
    fase_counter[m["fase"]] += 1

print(f"\nArchivos generados: {len(matches)}")
for fase, n in sorted(fase_counter.items()):
    print(f"  {fase:8s} -> {n}")

Output dir limpio: c:\Users\Administrador\Documents\MaestriaUSFQ\Procesamiento Lenguaje Natural\Proyecto Final\Corpus_Mundial\calendario-2026

Archivos generados: 104
  3rd      -> 1
  QF       -> 4
  R16      -> 8
  R32      -> 16
  SF       -> 2
  final    -> 1
  group    -> 72


## 6. Validacion

In [7]:
creados = sorted(OUTPUT_DIR.glob("*.md"))
print(f"Total archivos .md: {len(creados)}")
print("\nPrimeros 5:")
for p in creados[:5]:
    print(f"  {p.name}")
print("\nUltimos 5:")
for p in creados[-5:]:
    print(f"  {p.name}")

# Mostrar contenido de uno representativo (Argentina si existe)
arg = [p for p in creados if "argentina" in p.name.lower()]
if arg:
    print(f"\n=== Sample: {arg[0].name} ===")
    print(arg[0].read_text(encoding="utf-8"))

Total archivos .md: 104

Primeros 5:
  partido-001-mexico-vs-sudafrica.md
  partido-002-corea-del-sur-vs-republica-checa.md
  partido-003-canada-vs-bosnia-y-herzegovina.md
  partido-004-estados-unidos-vs-paraguay.md
  partido-005-haiti-vs-escocia.md

Ultimos 5:
  partido-100-cuartos-de-final.md
  partido-101-semifinales.md
  partido-102-semifinales.md
  partido-103-tercer-puesto.md
  partido-104-final.md

=== Sample: partido-019-argentina-vs-argelia.md ===
---
titulo: Argentina vs Argelia — Fase de Grupos, Grupo J
tema: calendario-mundial-2026
tipo: partido
fuente: Plataforma 91 / FIFA
partido_id: 19
edicion: 2026
fase: group
fase_es: Fase de Grupos
grupo: J
fecha: 2026-06-16
hora_local: '20:00'
hora_utc: '01:00'
estadio: Arrowhead Stadium
ciudad: Kansas City, MO
pais: USA
equipo_local: Argentina
equipo_visitante: Argelia
equipo_local_original: Argentina
equipo_visitante_original: Algeria
tags:
  - mundial-2026
  - calendario
  - partido
  - group
  - argentina
  - argelia
  - arrowhea

## 7. Siguiente paso

Re-correr `04-build-index.ipynb` para indexar los nuevos docs (~$0.05 USD).